In [1]:
import torch
from torch.utils.data import Dataset

In [2]:
# --- Custom transforms (callable objects) ---
class Normalize:
    """Normalize tensor to mean=0, std=1."""
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, x):
        return (x - self.mean) / self.std

class ClipValues:
    """Clip values to [min_val, max_val]."""
    def __init__(self, min_val=0.0, max_val=1.0):
        self.min_val = min_val
        self.max_val = max_val

    def __call__(self, x):
        return torch.clamp(x, self.min_val, self.max_val)

class Compose:
    """Chain multiple transforms together."""
    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, x):
        for t in self.transforms:
            x = t(x)
        return x

# --- Build a pipeline ---
pipeline = Compose([
    Normalize(mean=0.5, std=0.5),   # center around 0
    ClipValues(-3.0, 3.0),          # prevent extreme outliers
])

In [3]:
# Apply to sample data
raw_data = torch.rand(10) * 10  # values in [0, 10)
transformed = pipeline(raw_data)
print(f"Raw:        mean={raw_data.mean():.3f}  range=[{raw_data.min():.3f}, {raw_data.max():.3f}]")
print(f"Transformed: mean={transformed.mean():.3f}  range=[{transformed.min():.3f}, {transformed.max():.3f}]")

Raw:        mean=6.539  range=[0.979, 9.670]
Transformed: mean=2.796  range=[0.958, 3.000]


In [4]:
# --- Integrate transforms into a Dataset ---
class TransformedDataset(Dataset):
    """Dataset that applies transforms on the fly."""
    def __init__(self, features, labels, transform=None):
        self.features = features
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        x = self.features[idx]
        if self.transform:
            x = self.transform(x)
        return x, self.labels[idx]

# Usage
X = torch.randn(50, 4) * 5  # unnormalized
y = torch.randint(0, 2, (50,))
dataset = TransformedDataset(X, y, transform=Normalize(mean=X.mean(dim=0), std=X.std(dim=0)))

sample_x, sample_y = dataset[0]
print(f"\nTransformed sample: mean≈{sample_x.mean():.3f}  std≈{sample_x.std():.3f}")


Transformed sample: mean≈-0.103  std≈0.563
